# Image-to-Image генерация
## Stable Diffusion + Face Restoration
### Сохранение на Google Диск

In [ ]:
# Подключение Google Диска
from google.colab import drive
drive.mount("/content/drive")

# Папка для результатов (создаётся на Диске)
OUTPUT_DIR = "/content/drive/MyDrive/ColabOutputs/img2img"
!mkdir -p "{OUTPUT_DIR}"
print(f"Результаты будут в: {OUTPUT_DIR}")

In [ ]:
# Установка
!pip install diffusers transformers accelerate safetensors pillow requests
!pip install -q git+https://github.com/sczhou/CodeFormer.git
!pip install -q basicsr facexlib realesrgan

In [ ]:
import torch
from diffusers import StableDiffusionImg2ImgPipeline
from PIL import Image
from datetime import datetime
from IPython.display import display
import os, sys

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Устройство: {device}")
if device == "cuda":
    gpu = torch.cuda.get_device_properties(0)
    print(f"GPU: {gpu.name}, VRAM: {gpu.total_memory/1024**3:.1f} GB")

In [ ]:
# === ВЫБОР МОДЕЛИ ===
# 1 - DreamShaper (лучшие лица, универсальная) ★
# 2 - Realistic Vision (фотореалистичные лица)
# 3 - Protogen (тёмный стиль, без цензуры)
# 4 - Anything v5 (аниме/арт, идеальные лица)

MODEL = 1

models = {
    1: "Lykon/dreamshaper-8",
    2: "SG161222/Realistic_Vision_V6.0_B1_noVAE",
    3: "darkstorm2150/Protogen_x3.4_Official_Release",
    4: "Linaqruf/anything-v5.0",
}

model_id = models[MODEL]

pipe = StableDiffusionImg2ImgPipeline.from_pretrained(
    model_id,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32,
    safety_checker=None,
    requires_safety_checker=False
).to(device)

print(f"Загружена: {model_id}")

In [ ]:
# === ЗАГРУЗКА ИЗОБРАЖЕНИЯ ===
from google.colab import files
uploaded = files.upload()
filename = list(uploaded.keys())[0]
init_image = Image.open(filename).convert("RGB")
W, H = init_image.size
if W > 768 or H > 768:
    init_image.thumbnail((768, 768))
display(init_image)
print(f"Размер: {init_image.size}")

In [ ]:
# === ГЕНЕРАЦИЯ + СОХРАНЕНИЕ НА ДИСК ===
prompt = ""
negative_prompt = "mutated hands, bad face, deformed, blurry, bad anatomy"
strength = 0.6
steps = 30
guidance = 7.0

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
result = pipe(
    prompt=prompt,
    negative_prompt=negative_prompt,
    image=init_image,
    strength=strength,
    num_inference_steps=steps,
    guidance_scale=guidance
).images[0]

display(result)
out_path = f"{OUTPUT_DIR}/{timestamp}_output.png"
result.save(out_path)
print(f"✅ Сохранено: {out_path}")

In [ ]:
# === FACE RESTORATION (CodeFormer) ===
RESTORE_FACE = True

if RESTORE_FACE:
    sys.path.append("/usr/local/lib/python3.*/site-packages")
    from codeformer_codeformer import CodeFormer
    from basicsr.archs.rrdbnet_arch import RRDBNet
    from realesrgan import RealESRGANer

    codeformer = CodeFormer(
        model_path="CodeFormer/weights/CodeFormer/codeformer.pth",
        upscale=1,
        fidelity=0.7
    )

    img = result.copy()
    restored = codeformer.enhance(img, has_aligned=False, only_center=False)[0]
    display(restored)
    restored_path = f"{OUTPUT_DIR}/{timestamp}_restored.png"
    restored.save(restored_path)
    print(f"✅ Сохранено: {restored_path}")
else:
    print("Пропущено")

In [ ]:
# Ссылка на папку с результатами
from IPython.display import HTML
display(HTML(f'<a href="https://drive.google.com/drive/folders/1">Открыть Google Диск → ColabOutputs/img2img</a>'))
print(f"Все файлы: {OUTPUT_DIR}/")

### Как это работает:
- Результаты сохраняются сразу на Google Диск в `ColabOutputs/img2img/`
- Каждый файл с таймштампом — ничего не перезаписывается
- Открой приложение Google Диск на телефоне → там все готовые картинки
- Качать через `files.download()` больше не нужно

### Советы для лиц:
- **Prompt**: `detailed face, perfect eyes, sharp facial features, high detail face`
- **Negative**: `mutated hands, bad face, deformed, blurry, bad anatomy, disfigured, ugly, worst quality`
- **Модели**: DreamShaper-8 (№1) — лучшие лица, Realistic Vision (№2) — фото
- **CodeFormer** чинит кривые лица автоматически